# Paper Final Benchmark (Colab H100)

This notebook is for your paper-grade benchmark run.

It does four things:
1. Mounts Google Drive and enters the repo.
2. Creates a run-specific config from `configs/paper_final_500.toml`.
3. Builds graphs (if needed).
4. Runs 500-epoch benchmarking across `ff_layerwise`, `ff_e2e`, and `backprop`, then writes a paper summary table.
Implementation checks in this notebook:
- verifies benchmark outputs include fold stability columns (mean/std/min) for economics
- validates critic-aware FF benchmark metadata columns before generating paper summary


In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)


def _resolve_repo_root() -> Path:
    env_repo = os.environ.get("FRM_REPO_DIR", "").strip()
    if env_repo:
        p = Path(env_repo)
        if (p / "configs/default.toml").exists():
            return p

    candidates = [
        Path.cwd(),
        Path("/content/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/forward-risk-manager"),
    ]
    if IN_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            candidates.extend(
                sorted(p for p in drive_root.glob("*Forward*Risk*Manager*") if p.is_dir())
            )

    for p in candidates:
        if (p / "configs/default.toml").exists():
            return p

    raise FileNotFoundError(
        "Could not find repo root containing configs/default.toml. "
        "Set FRM_REPO_DIR or update candidate paths in this cell."
    )


ROOT = _resolve_repo_root()
os.chdir(ROOT)
print("repo root:", ROOT)
print("cwd:", Path.cwd())


In [ ]:
import importlib.util
import os
import re
import shlex
import subprocess
import sys
import time
from collections import deque

PYTHON_EXE = shlex.quote(sys.executable)


def run(cmd: str, allow_fail: bool = False, tail_lines: int = 200) -> bool:
    print("\n" + "=" * 120)
    print(cmd)
    print("=" * 120)
    t0 = time.time()

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

    logs_dir = ROOT / "runs" / "experiments" / "_paper_logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", cmd).strip("_")[:120] or "command"
    log_path = logs_dir / f"{int(time.time())}_{safe_name}.log"

    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None

    tail = deque(maxlen=max(40, int(tail_lines)))
    with log_path.open("w", encoding="utf-8") as lf:
        for line in proc.stdout:
            print(line, end="")
            lf.write(line)
            tail.append(line.rstrip("\n"))
    rc = proc.wait()

    elapsed = time.time() - t0
    print(f"\ncompleted in {elapsed:.2f}s | log: {log_path}")
    if rc != 0:
        if tail:
            print("---- command output tail ----")
            for ln in tail:
                print(ln)
            print("---- end tail ----")
        msg = f"command failed ({rc}): {cmd}"
        if allow_fail:
            print("WARNING:", msg)
            return False
        raise RuntimeError(msg)
    return True


required_modules = [
    "torch",
    "torch_geometric",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
INSTALL_DEPS = bool(missing)

if INSTALL_DEPS:
    print("Missing modules detected:", missing)
    run(f"{PYTHON_EXE} -m pip install --upgrade pip setuptools wheel")
    run(f"{PYTHON_EXE} -m pip install -r requirements.txt")
    run(f"{PYTHON_EXE} -m pip install -e .")
else:
    print("Dependencies already available. Skipping install.")


In [ ]:
from datetime import datetime, timezone
import re

TEMPLATE_CONFIG = ROOT / "configs" / "paper_final_500.toml"
assert TEMPLATE_CONFIG.exists(), f"Missing template config: {TEMPLATE_CONFIG}"

BASE_TOKEN = "runs/experiments/paper_final_500_TEMPLATE"
RUN_ID_OVERRIDE = ""  # leave empty for a fresh run
RESUME_POLICY = "auto"  # "auto" | "force_new" | "force_resume"

resume_requested = bool(RUN_ID_OVERRIDE.strip()) and RESUME_POLICY != "force_new"
if RESUME_POLICY == "force_resume" and not RUN_ID_OVERRIDE.strip():
    raise ValueError("RESUME_POLICY='force_resume' requires a non-empty RUN_ID_OVERRIDE")

if resume_requested:
    RUN_ID = RUN_ID_OVERRIDE.strip()
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID
    runtime_config = RUN_ROOT / "runtime_config.toml"
    if runtime_config.exists():
        print("resuming run id:", RUN_ID)
        print("runtime config:", runtime_config)
        print("run root:", RUN_ROOT)
    elif RESUME_POLICY == "force_resume":
        raise FileNotFoundError(f"Missing runtime config for resume: {runtime_config}")
    else:
        print(f"Resume requested but runtime config missing at {runtime_config}; creating a new run.")
        resume_requested = False

if not resume_requested:
    RUN_ID = f"paper_final_500_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

    for sub in ("data", "metrics", "plots", "logs", "models", "configs"):
        (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

    runtime_config = RUN_ROOT / "runtime_config.toml"
    cfg_txt = TEMPLATE_CONFIG.read_text()
    assert BASE_TOKEN in cfg_txt, f"Expected token {BASE_TOKEN!r} in template config"
    cfg_txt = cfg_txt.replace(BASE_TOKEN, f"runs/experiments/{RUN_ID}")
    runtime_config.write_text(cfg_txt)

    print("run id:", RUN_ID)
    print("runtime config:", runtime_config)
    print("run root:", RUN_ROOT)

APPLY_GPU_RUNTIME_OVERRIDES = True
PAPER_EXPECTED_GPU_PROFILE = "h100"
PAPER_ENFORCE_EXPECTED_GPU = False

if APPLY_GPU_RUNTIME_OVERRIDES:
    try:
        import torch

        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
    except Exception:
        gpu_name = ""

    gpu_upper = gpu_name.upper()
    gpu_profile = "h100" if "H100" in gpu_upper else ("a100" if "A100" in gpu_upper else ("t4" if "T4" in gpu_upper else "generic"))
    if gpu_profile != PAPER_EXPECTED_GPU_PROFILE:
        msg = (
            f"paper_final_benchmark_colab expects GPU profile '{PAPER_EXPECTED_GPU_PROFILE}' "
            f"but found '{gpu_profile}' ({gpu_name or 'unknown'})."
        )
        if PAPER_ENFORCE_EXPECTED_GPU:
            raise RuntimeError(msg)
        print("WARNING:", msg)

    amp_dtype = "bfloat16"
    auto_tune_max_batch = 128

    cfg_txt = runtime_config.read_text()
    cfg_txt = re.sub(r'(?m)^amp_dtype\s*=\s*".*"\s*$', f'amp_dtype = "{amp_dtype}"', cfg_txt)
    cfg_txt = re.sub(r'(?m)^backprop_amp_dtype\s*=\s*".*"\s*$', f'backprop_amp_dtype = "{amp_dtype}"', cfg_txt)
    cfg_txt = re.sub(r'(?m)^auto_tune_max_batch\s*=\s*\d+\s*$', f'auto_tune_max_batch = {auto_tune_max_batch}', cfg_txt)
    runtime_config.write_text(cfg_txt)

    print("gpu:", gpu_name or "<unknown>", "| profile:", gpu_profile)
    print("runtime overrides:", {"amp_dtype": amp_dtype, "auto_tune_max_batch": auto_tune_max_batch})


In [ ]:
graphs_path = RUN_ROOT / "data" / "graphs.pt"
RUN_BUILD_GRAPHS = not graphs_path.exists()
FORCE_REBUILD_GRAPHS = False

if FORCE_REBUILD_GRAPHS:
    RUN_BUILD_GRAPHS = True

if RUN_BUILD_GRAPHS:
    run(
        f"{PYTHON_EXE} -u scripts/build_graphs.py --config {shlex.quote(str(runtime_config))}"
    )
else:
    print(f"Skipping graph build; existing graphs found at {graphs_path}")


In [ ]:
import csv
import re

BENCH_MODES = ["ff_layerwise", "ff_e2e", "backprop"]
RESUME_BENCHMARK = True
FORCE_RERUN_MODES = set()  # e.g. {"backprop"}

benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
baseline_csv = RUN_ROOT / "metrics" / "benchmark_baseline.csv"

per_mode_benchmark = {
    mode: RUN_ROOT / "metrics" / f"benchmark_{mode}.csv" for mode in BENCH_MODES
}
per_mode_folds = {
    mode: RUN_ROOT / "metrics" / f"benchmark_walk_forward_folds_{mode}.csv" for mode in BENCH_MODES
}
per_mode_baseline = {
    mode: RUN_ROOT / "metrics" / f"benchmark_baseline_{mode}.csv" for mode in BENCH_MODES
}

cfg_template = runtime_config.read_text()
assert "out_csv" in cfg_template and "walk_forward_out_csv" in cfg_template, (
    "runtime config missing benchmark out paths"
)

(RUN_ROOT / "configs").mkdir(parents=True, exist_ok=True)

for mode in BENCH_MODES:
    mode_csv = per_mode_benchmark[mode]
    mode_folds = per_mode_folds[mode]
    mode_baseline = per_mode_baseline[mode]

    if (
        RESUME_BENCHMARK
        and mode not in FORCE_RERUN_MODES
        and mode_csv.exists()
        and mode_csv.stat().st_size > 0
    ):
        print(f"Skipping mode={mode}; found {mode_csv}")
        continue

    cfg_mode = cfg_template
    cfg_mode = re.sub(
        r'(?m)^out_csv\s*=\s*".*"\s*$',
        f'out_csv = "{mode_csv.as_posix()}"',
        cfg_mode,
        count=1,
    )
    cfg_mode = re.sub(
        r'(?m)^walk_forward_out_csv\s*=\s*".*"\s*$',
        f'walk_forward_out_csv = "{mode_folds.as_posix()}"',
        cfg_mode,
        count=1,
    )
    if re.search(r'(?m)^baseline_out_csv\s*=\s*".*"\s*$', cfg_mode):
        cfg_mode = re.sub(
            r'(?m)^baseline_out_csv\s*=\s*".*"\s*$',
            f'baseline_out_csv = "{mode_baseline.as_posix()}"',
            cfg_mode,
            count=1,
        )
    else:
        cfg_mode = re.sub(
            r'(?m)^out_csv\s*=\s*".*"\s*$',
            f'out_csv = "{mode_csv.as_posix()}"\nbaseline_out_csv = "{mode_baseline.as_posix()}"',
            cfg_mode,
            count=1,
        )

    cfg_mode_path = RUN_ROOT / "configs" / f"runtime_config_{mode}.toml"
    cfg_mode_path.write_text(cfg_mode)

    run(
        f"{PYTHON_EXE} -u scripts/benchmark_training.py "
        f"--config {shlex.quote(str(cfg_mode_path))} "
        f"--modes {mode}"
    )


def combine_csv(paths, out_path):
    rows = []
    fieldnames = []
    for p in paths:
        if not p.exists() or p.stat().st_size == 0:
            continue
        with p.open("r", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames:
                for k in reader.fieldnames:
                    if k not in fieldnames:
                        fieldnames.append(k)
            for row in reader:
                rows.append(row)

    if not rows:
        raise RuntimeError(f"No rows found to combine into {out_path}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    print(f"Wrote combined CSV: {out_path} rows={len(rows)}")


combine_csv([per_mode_benchmark[m] for m in BENCH_MODES], benchmark_csv)
combine_csv([per_mode_folds[m] for m in BENCH_MODES], folds_csv)
if any(p.exists() and p.stat().st_size > 0 for p in per_mode_baseline.values()):
    combine_csv([per_mode_baseline[m] for m in BENCH_MODES], baseline_csv)
else:
    print("No per-mode baseline CSVs found; skipping combined baseline file.")


In [ ]:
benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
baseline_csv = RUN_ROOT / "metrics" / "benchmark_baseline.csv"
summary_md = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
summary_csv = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
summary_json = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

import pandas as pd

assert benchmark_csv.exists(), f"Missing benchmark CSV: {benchmark_csv}"
bench_df = pd.read_csv(benchmark_csv)
required_cols = [
    "mode",
    "row_type",
    "objective_track",
    "econ_sharpe_uplift",
    "econ_sharpe_uplift_std",
    "econ_sharpe_uplift_min",
    "risk_head_enabled_effective",
    "portfolio_head_enabled_effective",
    "econ_regime_gate_enabled",
    "econ_regime_confidence_mean",
    "econ_regime_exposure_mean",
]
missing = [c for c in required_cols if c not in bench_df.columns]
assert not missing, f"Benchmark CSV missing expected columns: {missing}"
print("benchmark column check passed:", required_cols)

recommended_cols = [
    "econ_oos_sharpe_uplift_min",
    "econ_oos_folds_used",
    "econ_signal_polarity",
    "econ_regime_thresholding_enabled",
]
missing_recommended = [c for c in recommended_cols if c not in bench_df.columns]
if missing_recommended:
    print("WARNING: benchmark is missing newer econ diagnostics columns:", missing_recommended)

run(
    f"{PYTHON_EXE} -u scripts/paper_benchmark_summary.py "
    f"--benchmark {shlex.quote(str(benchmark_csv))} "
    f"--folds {shlex.quote(str(folds_csv))} "
    f"--out-md {shlex.quote(str(summary_md))} "
    f"--out-csv {shlex.quote(str(summary_csv))} "
    f"--out-json {shlex.quote(str(summary_json))}"
)

print("benchmark:", benchmark_csv)
print("folds:", folds_csv)
print("baseline:", baseline_csv, "exists=", baseline_csv.exists())
print("summary md:", summary_md)
print("summary csv:", summary_csv)
print("summary json:", summary_json)


In [6]:
from IPython.display import Markdown, display

if summary_md.exists():
    display(Markdown(summary_md.read_text()))
else:
    print("Missing summary markdown:", summary_md)

print("\nArtifact check:")
for path in [benchmark_csv, folds_csv, baseline_csv, summary_md, summary_csv, summary_json]:
    print(f"{path} -> exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")